In [ ]:
#this cells will have Load sheets + Inspect missingness
#if 100% missing -> safe to drop if it has values dont drop.
import pandas as pd

file_path = ("../../data/raw/metadata_model/Book1.xlsx")
meta_file = pd.ExcelFile(file_path)

header_rows = {
    "COMMUNICATION": 2,
    "CLEARANCE": 0,
    "AOsMEMOs": 2,
    "LEAVES": 0,
}

datasets={}


for sheet_name in meta_file.sheet_names:
    header_row = header_rows.get(sheet_name, 0)

    df = pd.read_excel(
        meta_file,
        sheet_name=sheet_name,
        header=header_row,
    )
    df.columns = df.columns.astype(str).str.strip()
    datasets[sheet_name] = df

    # Calculate missing percentages
    missing = df.isna().mean().sort_values(ascending=False)

    #Filter to show only columns  that actually have missing values (> 0%)
    missing_only = missing[missing > 0].sort_values(ascending=False)

    print(f"\n---{sheet_name} (Top Missing Columns %) ---")
    if missing_only.empty:
        print("No missing values found!")
    else:
        display(missing_only.head(10).to_frame(name="Missing %"))


        
            



---COMMUNICATION (Top Missing Columns %) ---


,Missing %
Unnamed: 9,1.000000
Unnamed: 10,0.998687
Unnamed: 11,0.998687
REMARKS 2,0.980306
REMARKS,0.689716
OFFICE/UNIT CONCERNED,0.074398
OFFICE/PERSON FROM,0.029322
DATE,0.000875



---CLEARANCE (Top Missing Columns %) ---


,Missing %
Unnamed: 12,1.000000
Unnamed: 11,1.000000
Unnamed: 13,1.000000
Unnamed: 14,1.000000
Unnamed: 10,1.000000
Unnamed: 9,1.000000
Unnamed: 15,0.998575
REMARKS 2,0.991453
C/O,0.056980
RELEASING DATE,0.034188



---AOsMEMOs (Top Missing Columns %) ---


,Missing %
REMARKS,0.428571



---LEAVES (Top Missing Columns %) ---


,Missing %
REMARKS,0.937595
DATE RECEIVED,0.068493
DAY/S APPLIED,0.068493
TRANSMITAL DATE,0.066971
STATUS,0.057839
DATE FILED,0.009132
TYPE OF LEAVE,0.007610
NAME,0.006088
CONTROL#,0.006088


In [104]:
#class Balance + schema audit
#spotting class imblance (can a model lean all categories fairly?).
#Identifying Schema Consistency accross Document Types
#Auditing Data Type and Parsing Quircks
#Estabilishing a Data Cleaning Roadmap
if not datasets:
    raise ValueError("The datasets are not loaded")

class_summary = []

for name, df in datasets.items():
    class_summary.append({
        "Document Class": name,
        "Total Rows (Sample)": len(df),
        "Total Columns": df.shape[1], #display these [rows[0], columns[1]] but since it is [1] it displays only columns
    })

summary_df = pd.DataFrame(class_summary)

total_rows = summary_df["Total Rows (Sample)"].sum()

if total_rows > 0:
    summary_df["Class Share (%)"] = (
        (summary_df["Total Rows (Sample)"] / total_rows) * 100
    ).round(2) #Keep two decimal places.

else:
    summary_df["Class Share (%)"] = 0.0

print("=== CLASS BALANCE SUMMARY ===")
display(summary_df.sort_values(by="Total Rows (Sample)", ascending=False))

print( "\n=== COLUMN SCHEMAS ===")
for name, df in datasets.items():
    print(f"\nSheet: {name}")
    schema_view =pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.astype(str),
        "Non-Null Count": df.notna().sum().values
    })
    
    display(schema_view.head(10))


=== CLASS BALANCE SUMMARY ===


,Document Class,Total Rows (Sample),Total Columns,Class Share (%)
0,COMMUNICATION,2285,12,62.47
1,CLEARANCE,702,16,19.19
3,LEAVES,657,9,17.96
2,AOsMEMOs,14,6,0.38



=== COLUMN SCHEMAS ===

Sheet: COMMUNICATION


,Column,Dtype,Non-Null Count
DATE,DATE,object,2283
CONTROL#,CONTROL#,str,2285
SUBJECT,SUBJECT,str,2285
DESCRIPTION,DESCRIPTION,str,2285
TYPE,TYPE,str,2285
OFFICE/PERSON FROM,OFFICE/PERSON FROM,str,2218
OFFICE/UNIT CONCERNED,OFFICE/UNIT CONCERNED,str,2115
REMARKS,REMARKS,str,709
REMARKS 2,REMARKS 2,str,45
Unnamed: 9,Unnamed: 9,float64,0



Sheet: CLEARANCE


,Column,Dtype,Non-Null Count
DATE,DATE,datetime64[us],702
CONTROL#,CONTROL#,str,702
FULL N AME,FULL N AME,str,702
OFFICE,OFFICE,str,690
PURPOSE,PURPOSE,str,693
REMARKS,REMARKS,str,679
RELEASING DATE,RELEASING DATE,datetime64[us],678
C/O,C/O,str,662
REMARKS 2,REMARKS 2,str,6
Unnamed: 9,Unnamed: 9,float64,0



Sheet: AOsMEMOs


,Column,Dtype,Non-Null Count
DATE RECEIVED,DATE RECEIVED,datetime64[us],14
DOCUMENT DATE,DOCUMENT DATE,datetime64[us],14
AO/MEMO/EO #,AO/MEMO/EO #,str,14
SUBJECT,SUBJECT,str,14
OFFICE/UNIT/PERSON FROM,OFFICE/UNIT/PERSON FROM,str,14
REMARKS,REMARKS,str,8



Sheet: LEAVES


,Column,Dtype,Non-Null Count
DATE FILED,DATE FILED,datetime64[us],651
TRANSMITAL DATE,TRANSMITAL DATE,datetime64[us],613
CONTROL#,CONTROL#,str,653
NAME,NAME,str,653
TYPE OF LEAVE,TYPE OF LEAVE,str,652
REMARKS,REMARKS,str,41
DAY/S APPLIED,DAY/S APPLIED,str,612
DATE RECEIVED,DATE RECEIVED,datetime64[us],612
STATUS,STATUS,str,619


In [108]:
#check unnamed columns and remove 

#do this first
for name, df in datasets.items():
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]

    for col in unnamed:
        print(name, col, df[col].notna().sum())

#After confirming that, you can add the cleaning step in the same cell:
for name, df in datasets.items():
    mask = ~df.columns.str.startswith("Unnamed")
    datasets[name] = df.loc[:, mask]


In [144]:
#Inspect the fields that might become a model inputs
#the question here is If I hide the sheet name and only show this text, can this text reasonably help identify the document class?
candidate_columns = {
    "COMMUNICATION": ["SUBJECT", "DESCRIPTION"],
    "CLEARANCE":["PURPOSE"] ,
    "AOsMEMOs": ["AO/MEMO/EO #", "SUBJECT"],
    "LEAVES":["TYPE OF LEAVE", "REMARKS", "DAY/S APPLIED"],
}

for name, cols in candidate_columns.items():
    print(f"\n---{name}---")
    display(datasets[name][cols].sample(10, random_state=42))


---COMMUNICATION---


,SUBJECT,DESCRIPTION
2094,FOR WITHDRAWAL OF RETENTION MONEY: REHABILITAT...,FOR WITHDRAWAL OF RETENTION MONEY: REHABILITAT...
764,Letter from RSC RO2: 16TH NATIONAL CONVENTION ...,Letter from RSC RO2: 16TH NATIONAL CONVENTION ...
240,NOTICE OF MEETING RE: PEO MONITORING SYSTEM,NOTICE OF MEETING RE: PEO MONITORING SYSTEM
2032,"1ST INDORSEMENT : JULY 7, 2026 RESPECTFULLY, F...","1ST INDORSEMENT : JULY 7, 2026 RESPECTFULLY, F..."
564,REQUEST FOR WRITE-OFF OF VARIOUS DORMANT RECEI...,REQUEST FOR WRITE-OFF OF VARIOUS DORMANT RECEI...
1576,"MEMORANDUM ORDER NO. 12, S.2026",LIFTING OF THE DRESS CODE AND IMPLEMENTATION O...
366,"AUTHORITY TO RENDER OVERTIME - ALL SATURDAYS, ...","AUTHORITY TO RENDER OVERTIME - ALL SATURDAYS, ..."
2114,SALARY ADJUSTMENT (NOSA) - PHO,SALARY ADJUSTMENT (NOSA) - PHO AND SP
940,"DESIGNATION OF MS. MARICAR LARDIZABAL, ACCOUNT...","DESIGNATION OF MS. MARICAR LARDIZABAL, ACCOUNT..."
1942,"STATEMENT OF ALLOTMENTS, OBLIGATIONS AND BALAN...","1. STATUS OF APPROPRIATION, ALLOTMENT AND OBLI..."



---CLEARANCE---


,PURPOSE
493,TRAVEL ABROAD
164,TERMINAL
54,RETIREMENT
641,"RETIREMENT EFFECTIVE SEPTEMBER 1, 2026"
607,"MATERNITY LEAVE (JULY 17, 2026)"
429,END OF CONTRACT
576,NaN
669,"TRAVEL ABROAD (THAILAND) OCTOBER 1 - 4, 2026"
636,"TRAVEL ABROAD (BANGKOK, THAILAND) JULY 26 - JU..."
411,TRAVEL ABROAD



---AOsMEMOs---


,AO/MEMO/EO #,SUBJECT
9,MEMO-2026-04-13-01,DOCUMENTARY REQUIREMENTS FOR REIMBURSEMENT OF ...
11,MEMO-2026-41,REITERATE THE SUBMISSION OF THE PPA FOR POSSIB...
0,EO-2026-01,EXECUTIVE ORDER NO. 01 - STREAMLINING AND RATI...
12,MEMO-2026-28,MEMORANDUM ORDER NO. 28 S. 2026 ADOPTION OF LA...
5,AO-2026-09,INSTITUTING THE STANDARD RATES FOR MEALS AND S...
8,MEMO-2026-34A,MEMORANDUM 34A: REQUEST FOR PERMANENT ALTERNAT...
2,AO-2026-1A,APPROVING AND ADOPTING THE JOINT RESOLUTION NO...
1,AO-2026-02,ORDER ADOPTING THE GUIDELINES ON THE USE AND O...
13,AO-2026-13,REORGANIZING THE ADMINISTRATIVE AND FINANCIAL ...
4,EO-2026-06,RECONSTITUTING THE COMPREHENSIVE DEVELOPMENT P...



---LEAVES---


,TYPE OF LEAVE,REMARKS,DAY/S APPLIED
627,MANDATORY,NaN,"SEPT. 2, 3, 4, 7, 8, 2026"
271,SICK LEAVE,NaN,"SEPTEMBER 5, 2025"
290,SPL,NaN,"SEPTEMBER 23, 2025"
63,SICK LEAVE,NaN,"FEB 5, 2025"
302,MONETIZATION,NaN,TEN DAYS
78,VACATION LEAVE,NaN,"MARCH 13 & 14, 2025"
644,SPL,NaN,"SEPTEMBER 8, 2026"
576,WELLNESS LEAVE,NaN,"JULY 15 & 16, 2026"
72,SICK LEAVE,NaN,"FEB 17, 2025"
221,COMPENSATORY,NaN,"JULY 14, 2025"


In [147]:
#cell 5 classify each candidate as KEEP, INVESTIGATE, or DROP based on the sampled values.
feature_audit = pd.DataFrame([
    ["COMMUNICATION", "SUBJECT", "KEEP", "Meaningful document text"],
    ["COMMUNICATION", "DESCRIPTION", "KEEP", "Meaningful document text"],
    ["COMMUNICATION", "TYPE", "REMOVED", "May encode workflow/category"],
    ["CLEARANCE", "PURPOSE", "KEEP", "Describes why clearance was issued"],
    ["AOsMEMOs", "AO/MEMO/EO #", "INVESTIGATE", "May directly reveal class"],
    ["AOsMEMOs", "AO/MEMO/EO #", "KEEP", "Meaningful document text"],
    ["LEAVES", "TYPE OF LEAVE", "INVESTIGATE", "Describe the reason for leave"],
    ["LEAVES", "REMARKS", "KEEP", "Potential semantic text"],
    ["LEAVES", "DAY/S APPLIED", "DROP", "Weak semantic value"],
], columns=["Class", "Column", "Decision", "Reason"])

display(feature_audit)

,Class,Column,Decision,Reason
0,COMMUNICATION,SUBJECT,KEEP,Meaningful document text
1,COMMUNICATION,DESCRIPTION,KEEP,Meaningful document text
2,COMMUNICATION,TYPE,REMOVED,May encode workflow/category
3,CLEARANCE,PURPOSE,KEEP,Describes why clearance was issued
4,AOsMEMOs,AO/MEMO/EO #,INVESTIGATE,May directly reveal class
5,AOsMEMOs,AO/MEMO/EO #,KEEP,Meaningful document text
6,LEAVES,TYPE OF LEAVE,INVESTIGATE,Describe the reason for leave
7,LEAVES,REMARKS,KEEP,Potential semantic text
8,LEAVES,DAY/S APPLIED,DROP,Weak semantic value


In [151]:
#candidate features
#Non-null count-how many rows actually contain data
#Cardinality — number of unique values in a field.
# Text length — number of characters in the text.
# Feature sparsity — how often a feature is missing.
# Categorical feature — a field with a small repeated set of values.
# Free-text feature — a field with many unique natural-language values.

for name, cols in candidate_columns.items():
    print(f"\n==={name}===")

    for col in cols:
        s = datasets[name][col]
        print(col)
        print("None-null:", s.notna().sum())
        print("Unique:", s.nunique(dropna=True))
        print("Avg length:", s.dropna().astype(str).str.len().mean())


===COMMUNICATION===
SUBJECT
None-null: 2285
Unique: 2120
Avg length: 104.01269146608315
DESCRIPTION
None-null: 2285
Unique: 2165
Avg length: 118.33129102844639

===CLEARANCE===
PURPOSE
None-null: 693
Unique: 179
Avg length: 20.362193362193363

===AOsMEMOs===
AO/MEMO/EO #
None-null: 14
Unique: 14
Avg length: 11.285714285714286
SUBJECT
None-null: 14
Unique: 14
Avg length: 208.64285714285714

===LEAVES===
TYPE OF LEAVE
None-null: 652
Unique: 22
Avg length: 9.757668711656441
REMARKS
None-null: 41
Unique: 20
Avg length: 13.21951219512195
DAY/S APPLIED
None-null: 612
Unique: 401
Avg length: 15.929738562091503


In [ ]:
#finalize the usable features and build a normalized document_text field.
